### 環境初始化

##### 掛載Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/vllm-inference-optimization
import os
print(f"✅ 成功切換至專案工作目錄: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/vllm-inference-optimization
✅ 成功切換至專案工作目錄: /content/drive/MyDrive/vllm-inference-optimization


##### Git

In [4]:
!git config --global user.email "walterlai@gapp.nthu.edu.tw"
!git config --global user.name "WalterOuO"

In [ ]:
# !git rm -r --cached . # 不要隨便用, 會把過去git紀錄全清空

# 如果要強制修改上一次push的所有file的commit：用--amend，push 尾巴加上 -f
# !git commit --amend -m "Initial Commit"

# 如果上一次push的file只有一個檔案要修改commit:
# !git reset HEAD~1 --soft   # 用 reset 把上一次的紀錄打開: HEAD~1就是上一次, HEAD~2就是上兩次
# !git reset HEAD .      # 把所有檔案先移出暫存區
# !git add .          # 再把上次push的所有檔案加進去
# !git reset somefile     # 唯獨把要修改commit的檔案移出暫存區
# !git commit -m "原本commit" # 把不用改的檔案commit就寫原本commit
# !git add app/.qdrant-initialized # 再把要改的檔案加入暫存區
# !git commit -m "new commit" # 再單獨修改它的新commit
# !git push -u origin main -f # 強制覆蓋線上的錯誤紀錄

In [ ]:
# feat	 新增功能 (Feature)
# fix	  修復 Bug (Bug Fix)
# docs	 僅修改文件 (Documentation)
# style	 不影響程式碼運作的格式調整（如空格、分號、缩排）
# refactor 輸出不變時重構程式碼（既不是新增功能，也不是修補 Bug）
# perf	 效能優化 (Performance)
# test	 增加或修改測試代碼 (Test)
# chore	 建構流程或輔助工具的變動（如 CI/CD、套件版號更新）
# revert	撤銷先前的 commit

In [5]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/vllm-inference-optimization/.git/


In [ ]:
!git add .
# !git commit --amend -m "Initial Commit"
!git commit -m "Initial commit."
!git push -u origin main

##### 環境檢查

In [ ]:
!nvidia-smi

import torch
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU 型號: {torch.cuda.get_device_name(0)}")
  print(f"VRAM 總容量: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

##### 安裝依賴套件

In [ ]:
!pip install -q -r requirements.txt

print("✅ 所有依賴套件已完成安裝！")

##### 執行Baseline實驗

In [ ]:
!python src/experiments/day1_awq_runner.py --mode baseline --config configs/day1_awq.yaml

##### 執行優化實驗

In [ ]:
!python src/experiments/day1_awq_runner.py --mode awq --config configs/day1_awq.yaml

##### 生成圖表對比

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# 讀取 JSON 結果
with open("results/raw_logs/day1_baseline.json", "r") as f:
    baseline_res = json.load(f)

with open("results/raw_logs/day1_awq.json", "r") as f:
    awq_res = json.load(f)

df = pd.DataFrame([baseline_res, awq_res])

# 印出文字版摘要對照表
print("================ Day 1 實驗對照總結 ================")
print(df[["mode", "model", "concurrency", "static_vram_gb", "peak_vram_gb", "system_tps"]].to_string(index=False))

# 繪製對比柱狀圖
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. System TPS 對比 (越大越好)
axes[0].bar(df['mode'], df['system_tps'], color=['#7293CB', '#E1974C'])
axes[0].set_title('System Total Throughput (TPS) 🚀\n(Higher is better)')
axes[0].set_ylabel('Tokens / Sec')
for i, v in enumerate(df['system_tps']):
    axes[0].text(i, v + 2, f"{v}", ha='center', fontweight='bold')

# 2. Peak VRAM 對比 (越小越好)
axes[1].bar(df['mode'], df['peak_vram_gb'], color=['#7293CB', '#84BA5B'])
axes[1].set_title('Peak VRAM Usage (GB) 📉\n(Lower is better)')
axes[1].set_ylabel('VRAM (GB)')
for i, v in enumerate(df['peak_vram_gb']):
    axes[1].text(i, v + 0.2, f"{v} GB", ha='center', fontweight='bold')

plt.tight_layout()
chart_path = "results/charts/day1_awq_comparison.png"
plt.savefig(chart_path)
plt.show()

print(f"\n📊 圖表已儲存至: {chart_path}")